# 02 Ingest Earthquake Hazard Feeds (USGS)

Stage: `01_ingest_hazard`
Discipline: earthquake hazard data generation.

Outputs:
- `JupyterNotebooks/outputs/index_pipeline/01_ingest/usgs_earthquake_events.csv`


In [1]:
# Cell 1: Setup
import importlib.util
import subprocess
import sys
import logging
import os
from datetime import datetime, timedelta, timezone
from pathlib import Path


def ensure_packages(packages):
    missing = [p for p in packages if importlib.util.find_spec(p) is None]
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])


ensure_packages(["pandas", "requests"])

import pandas as pd
import requests

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("index-pipeline-stage01-eq")


def find_repo_root():
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c / "JupyterNotebooks").exists():
            return c
    return p


REPO_ROOT = find_repo_root()
OUTPUT_DIR = REPO_ROOT / "JupyterNotebooks" / "outputs" / "index_pipeline" / "01_ingest"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    from IPython.display import display
except ImportError:
    display = print


In [2]:
# Cell 2: Configuration
LOOKBACK_DAYS = int(os.environ.get("EQ_LOOKBACK_DAYS", "30"))
MIN_MAG = float(os.environ.get("EQ_MIN_MAG", "1.0"))

PR_EXTENT = {
    "minlatitude": 17.4,
    "maxlatitude": 18.9,
    "minlongitude": -68.6,
    "maxlongitude": -65.0,
}

USGS_EQ_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"
RUN_UTC = datetime.now(timezone.utc)
START_UTC = RUN_UTC - timedelta(days=LOOKBACK_DAYS)

print(f"Lookback days: {LOOKBACK_DAYS} | Min magnitude: {MIN_MAG}")


Lookback days: 30 | Min magnitude: 1.0


In [3]:
# Cell 3: Fetch and export
params = {
    "format": "geojson",
    "starttime": START_UTC.strftime("%Y-%m-%d"),
    "endtime": RUN_UTC.strftime("%Y-%m-%d"),
    "minmagnitude": MIN_MAG,
    **PR_EXTENT,
    "orderby": "time",
}

resp = requests.get(USGS_EQ_URL, params=params, timeout=120)
resp.raise_for_status()
payload = resp.json()

rows = []
for feat in payload.get("features", []):
    props = feat.get("properties", {})
    geom = feat.get("geometry") or {}
    coords = geom.get("coordinates") or [None, None, None]
    event_time = pd.to_datetime(props.get("time"), unit="ms", utc=True, errors="coerce")

    rows.append({
        "event_id": feat.get("id"),
        "time_utc": event_time,
        "magnitude": props.get("mag"),
        "depth_km": coords[2],
        "longitude": coords[0],
        "latitude": coords[1],
        "place": props.get("place"),
        "status": props.get("status"),
        "alert_level": props.get("alert"),
        "tsunami": props.get("tsunami"),
        "updated_utc": pd.to_datetime(props.get("updated"), unit="ms", utc=True, errors="coerce"),
        "run_utc": RUN_UTC,
    })

eq_df = pd.DataFrame(rows).sort_values("time_utc", ascending=False).reset_index(drop=True)

eq_out = OUTPUT_DIR / "usgs_earthquake_events.csv"
eq_df.to_csv(eq_out, index=False)

print(f"Earthquake rows: {len(eq_df)}")
print(f"Output: {eq_out}")
display(eq_df.head(10))


Earthquake rows: 140
Output: /Users/sivamanisubrahmanyaharivamsipullipudi/Downloads/DEAN_690/Spring2026DEAN/Spring2026DAEN/JupyterNotebooks/JupyterNotebooks/outputs/index_pipeline/01_ingest/usgs_earthquake_events.csv


,event_id,time_utc,magnitude,depth_km,longitude,latitude,place,status,alert_level,tsunami,updated_utc,run_utc
0,pr71513248,2026-04-10 16:50:26.530000+00:00,2.09,6.88,-66.768500,18.060833,"1 km W of Santo Domingo, Puerto Rico",reviewed,None,0,2026-04-10 17:12:16.730000+00:00,2026-04-11 09:38:58.840699+00:00
1,pr71513243,2026-04-10 08:42:21.690000+00:00,2.90,66.35,-66.698500,18.694833,"24 km N of Arecibo, Puerto Rico",reviewed,None,0,2026-04-10 08:57:42.840000+00:00,2026-04-11 09:38:58.840699+00:00
2,pr71513238,2026-04-10 08:27:34.350000+00:00,2.36,14.51,-66.860167,17.840333,"15 km SSE of Guánica, Puerto Rico",reviewed,None,0,2026-04-10 08:41:14.990000+00:00,2026-04-11 09:38:58.840699+00:00
3,pr71513233,2026-04-10 08:12:24.180000+00:00,2.34,23.61,-67.154167,18.220167,"2 km NW of Mayagüez, Puerto Rico",reviewed,None,0,2026-04-10 08:35:33.010000+00:00,2026-04-11 09:38:58.840699+00:00
4,pr71513213,2026-04-10 06:45:11.270000+00:00,2.08,10.27,-66.917333,17.941667,"3 km SSW of Guánica, Puerto Rico",reviewed,None,0,2026-04-10 06:59:01.650000+00:00,2026-04-11 09:38:58.840699+00:00
5,pr71513198,2026-04-10 02:06:13.480000+00:00,2.74,15.12,-66.141667,17.935833,"3 km SE of Jobos, Puerto Rico",reviewed,None,0,2026-04-10 02:22:29.590000+00:00,2026-04-11 09:38:58.840699+00:00
6,pr71513188,2026-04-10 01:19:11.330000+00:00,1.51,14.14,-66.230833,18.071667,"1 km NE of Vázquez, Puerto Rico",reviewed,None,0,2026-04-10 01:32:08.630000+00:00,2026-04-11 09:38:58.840699+00:00
7,pr71513183,2026-04-10 01:17:31.670000+00:00,2.17,16.25,-67.137833,17.981667,"5 km E of Pole Ojea, Puerto Rico",reviewed,None,0,2026-04-10 01:32:29+00:00,2026-04-11 09:38:58.840699+00:00
8,pr71513173,2026-04-09 23:11:36.310000+00:00,1.62,5.69,-66.862500,18.068500,"3 km NNW of Yauco, Puerto Rico",reviewed,None,0,2026-04-09 23:21:53.110000+00:00,2026-04-11 09:38:58.840699+00:00
9,pr71513168,2026-04-09 19:31:48.090000+00:00,1.87,17.82,-66.845500,17.968000,"4 km SW of Indios, Puerto Rico",reviewed,None,0,2026-04-09 20:04:55.020000+00:00,2026-04-11 09:38:58.840699+00:00
